In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

columns = ["Rank", "Tournament Name", "Prize Pool", "Game", "Teams", "Number of Players"]

def get_tournament_details(row):
    cells = row.find_elements(By.TAG_NAME, "td")
    texts = [cell.text.strip() for cell in cells]

    # remove the text in the teams and players columns
    def clean_stat(text):
        return (
            text.replace("Teams", "")
                .replace("Players", "")
                .replace("\n", " ")
                .strip()
        )


    return {
        "Rank": texts[0],
        "Tournament Name": texts[1],
        "Prize Pool": texts[2],              
        "Game": texts[3],
        "Teams": clean_stat(texts[5]),                          # There is a gap between the table index of Game and Teams and I had to find out by trial and error
        "Number of Players": clean_stat(texts[6])
    }

def main():
    tournament_data = []

    driver = webdriver.Chrome()

    try:
        for page_id in range(1, 5):
            url = f"https://www.esportsearnings.com/tournaments?page={page_id}"
            driver.get(url)

            WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
            )

            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

            for row in rows:
                tournament = get_tournament_details(row)
                if tournament:
                    tournament_data.append(tournament)

                if len(tournament_data) >= 100:
                    break

            if len(tournament_data) >= 100:
                break

    finally:
        driver.quit()

    df = pd.DataFrame(tournament_data[:100], columns=columns)
    #save to csv file
    df.to_csv("top_100_tournaments_by_prize_pool.csv", index=False)

    print(df.head())
    print(f"Saved {len(df)} tournaments.")

if __name__ == "__main__":
    main()

  Rank         Tournament Name      Prize Pool    Game Teams Number of Players
0   1.  The International 2021  $40,018,400.00  Dota 2    18                90
1   2.  The International 2019  $34,330,069.00  Dota 2    18                90
2   3.  The International 2018  $25,532,177.00  Dota 2    18                90
3   4.  The International 2017  $24,687,919.00  Dota 2    18                90
4   5.  The International 2016  $20,770,460.00  Dota 2    16                80
Saved 100 tournaments.
